# Generate both full DC4 source catalog YAMLs

This notebook creates **two 71-source, full-observation YAML catalogs** used by the NGC 4151 fit: one with the parallel MEGAlib chunk convention and one with global (no-chunk) lightcurve normalization. Both start from the same injected source definitions. The parallel result follows the normalization in `Source_catalog_generator_all_DC4_lightcurve.ipynb`.

Run the cells from top to bottom. The notebook reads the `cosi-sim` source library, the full spacecraft orientation, and **existing** NSIDE-8 spatial templates. It does not read event FITS files, build detector responses, make comparison plots or tables, or write a response recipe.

The saved amplitudes are time averages over the **full elapsed observation**. A downstream fit needs a response recipe made with the *same* normalization method as its YAML catalog. The existing parallel-chunk `.response.json` recipe does not describe the no-chunk catalog.


## 1. Inputs and outputs

Edit these paths if your source library or DC4 files live elsewhere. The two default outputs have `_generated.yaml` suffixes so they will not replace the checked-in catalog. Set `OVERWRITE_CATALOGS = True` only when you intentionally want to replace existing files at the chosen output paths.


In [2]:
from pathlib import Path
from copy import deepcopy
import gzip
import re

import astropy.units as u
import numpy as np
from astromodels import (
    Asymm_Gaussian_on_sphere,
    Band,
    Cutoff_powerlaw,
    ExtendedSource,
    Gaussian,
    Gaussian_on_sphere,
    LinearPolarization,
    Model,
    PointSource,
    Powerlaw,
    SpectralComponent,
)

from cosipy.response.temporal_profile import LightcurveProfile
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat

NOTEBOOK_DIRECTORY = Path("/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN")
SOURCE_DATA_ROOT = Path("/Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library")
DC4_FILES_ROOT = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Radio_Quiet_AGN/DC4_Files"
)
ORIENTATION_PATH = DC4_FILES_ROOT / (
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
)
SPATIAL_TEMPLATE_DIRECTORY = DC4_FILES_ROOT / "DC4_Sources/DC4_Spatial_Templates"
OUTPUT_PARALLEL_CATALOG = NOTEBOOK_DIRECTORY / (
    "source_catalog_DC4_all_71files_full_elapsed_lightcurve_parallel_chunks_generated.yaml"
)
OUTPUT_NO_CHUNKS_CATALOG = NOTEBOOK_DIRECTORY / (
    "source_catalog_DC4_all_71files_full_elapsed_lightcurve_no_chunks_generated.yaml"
)
OVERWRITE_CATALOGS = True

# Spatial templates are inputs here; this notebook never creates FITS files.
SPATIAL_TEMPLATE_NSIDE = 8
SPATIAL_TEMPLATE_FORMAT_TAG = "perMeV_v3"

for input_path in (SOURCE_DATA_ROOT, ORIENTATION_PATH, SPATIAL_TEMPLATE_DIRECTORY):
    if not input_path.exists():
        raise FileNotFoundError(input_path)
OUTPUT_CATALOGS = {
    "parallel_chunks": OUTPUT_PARALLEL_CATALOG,
    "no_chunks": OUTPUT_NO_CHUNKS_CATALOG,
}
if len({path.resolve() for path in OUTPUT_CATALOGS.values()}) != len(OUTPUT_CATALOGS):
    raise ValueError("The two catalogs need distinct output paths")
for output_path in OUTPUT_CATALOGS.values():
    if output_path.exists() and not OVERWRITE_CATALOGS:
        raise FileExistsError(f"Output exists: {output_path}; choose a new path or enable overwrite")

## 2. Spectrum helpers

`SpecFromDat` preserves each injected table shape. We adjust its normalization so integrating from 100 keV to 10 MeV recovers the injected photon flux. Analytic components use the same energy band.

In [3]:
def find_source_file(challenge, filename):
    """Find one source-library file below DC3/ or DC4/."""
    search_root = SOURCE_DATA_ROOT / challenge
    matches = sorted(search_root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} below {search_root}. "
            "The simulated FITS source files are not spectral tables; copy the "
            "corresponding .dat file from the cosi-sim source library into this "
            "DC3/DC4 directory."
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Found more than one copy of {filename!r}: {matches}. "
            "Keep one copy or set the desired path explicitly."
        )

    return matches[0].resolve()


cygx1_low_path = find_source_file("DC3", "cygX1_hard_0.1-0.4_spec.dat")
cygx1_high_path = find_source_file("DC3", "cygX1_hard_0.4-10_spec.dat")
grs1758_path = find_source_file("DC3", "GRS1758_spec.dat")
crab_nebula_path = find_source_file("DC4", "Crab_Ne_spec.dat")
crab_p1_path = find_source_file("DC4", "Crab_P1_spec.dat")
crab_bridge_path = find_source_file("DC4", "Crab_Brg_spec.dat")
crab_p2_path = find_source_file("DC4", "Crab_P2_spec.dat")
four_c_71p07_path = find_source_file("DC4", "4C71p07_spectrum.dat")
four_c_21p35_noflare_path = find_source_file(
    "DC3", "4C_21_spectrum_noflare.dat"
)
four_c_21p35_flare_path = find_source_file(
    "DC3", "4C_21_spectrum_flare.dat"
)
maxi_j1820_low_path = find_source_file("DC3", "MAXIJ1820_0.1-0.4_spec.dat")
maxi_j1820_high_path = find_source_file("DC3", "MAXIJ1820_0.4-10_spec.dat")
maxi_j1348_low_path = find_source_file("DC3", "MAXIJ1348_0.1-0.4_spec.dat")
maxi_j1348_high_path = find_source_file("DC3", "MAXIJ1348_0.4-10_spec.dat")
three_c_454p3_low_path = find_source_file("DC4", "3C454p3_low_spectrum.dat")
three_c_454p3_high_path = find_source_file("DC4", "3C454p3_high_spectrum.dat")
cyg_x3_path = find_source_file("DC4", "cygX3_transition_spec.dat")
psr_b1259_path = find_source_file("DC3", "PSRB1259_spec.dat")
ls_5039_path = find_source_file("DC3", "LS5039_spec.dat")
psr_j1846_path = find_source_file("DC3", "J1846_spec.dat")
co_nova_continuum_path = find_source_file("DC3", "nova_co_continuum_spec.dat")

In [4]:
ENERGY_MIN = 100.0
ENERGY_MAX = 10_000.0
DIFFERENTIAL_FLUX_UNIT = 1 / (u.keV * u.cm**2 * u.s)


def table_spectrum(dat_path, injected_photon_flux):
    """Create a SpecFromDat component normalized to an integrated flux."""
    spectrum = SpecFromDat(
        K=float(injected_photon_flux),
        dat=Path(dat_path).resolve(),
    )

    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    current_integral = np.trapezoid(spectrum(energy), energy)

    if not np.isfinite(current_integral) or current_integral <= 0:
        raise ValueError(f"Invalid spectrum integral for {dat_path}: {current_integral}")

    # `SpecFromDat` normalizes the table internally; correct its numerical integral.
    spectrum.K.value *= float(injected_photon_flux) / current_integral
    return spectrum


def set_normalization_step(parameter):
    """Use a scale-appropriate initial optimizer step."""
    parameter.delta = max(abs(parameter.value) * 0.05, 1e-12)

## 3. Reviewed continuum sources

The first 21 entries retain the individually reviewed DC4 and DC3 spectra, polarizations, positions, and injected fluxes. NGC 4151 uses the updated cutoff power law plus power-law tail. The remaining 50 entries are parsed directly from their MEGAlib `.source` files in the next section.

In [5]:
# Cyg X-1: keep both injected energy-range tables.
cygx1_low_flux = 0.04243172227636306
cygx1_high_flux = 0.003371822180706115

cygx1_low = table_spectrum(cygx1_low_path, cygx1_low_flux)
cygx1_high = table_spectrum(cygx1_high_path, cygx1_high_flux)
cygx1_spectrum = cygx1_low + cygx1_high

cygx1 = PointSource(
    "cyg_x1_hard",
    l=71.33496,
    b=3.066917,
    spectral_shape=cygx1_spectrum,
)

In [6]:
# Crab: preserve nebula and three pulse components with their polarization.
crab_nebula = table_spectrum(crab_nebula_path, 0.033197515)
crab_p1 = table_spectrum(crab_p1_path, 0.002380673782066656)
crab_bridge = table_spectrum(crab_bridge_path, 0.0006280494007254569)
crab_p2 = table_spectrum(crab_p2_path, 0.0032012384701172636)

crab_components = [
    SpectralComponent(
        "nebula",
        crab_nebula,
        LinearPolarization(40.0, 160.0),
    ),
    SpectralComponent(
        "peak1",
        crab_p1,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "bridge",
        crab_bridge,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "peak2",
        crab_p2,
        LinearPolarization(20.0, 145.0),
    ),
]

crab = PointSource(
    "crab",
    l=184.5575,
    b=-5.78434,
    components=crab_components,
)

for component in crab.components.values():
    component.polarization.degree.fix = True
    component.polarization.angle.fix = True

In [7]:
# 1E 1740.7−2942: read the injected table and position from its source file.
definition = find_source_file("DC3", "1E1740_compow.source")
properties = {}
for raw_line in definition.read_text().splitlines():
    fields = raw_line.split("#", 1)[0].split()
    if fields and "." in fields[0]:
        properties[fields[0].split(".", 1)[1]] = fields[1:]
assert properties["Spectrum"][0].lower() == "file"
spectrum = table_spectrum(
    definition.parent / properties["Spectrum"][1], float(properties["Flux"][0]),
)
latitude, longitude = map(float, properties["Orientation"][-2:])
one_e_1740 = PointSource("one_e_1740_compow", l=longitude % 360.0, b=latitude, spectral_shape=spectrum)

In [8]:
# GRS 1758−258: preserve the injected Comptonization table.
grs1758_flux = 0.003495
grs1758_spectrum = table_spectrum(grs1758_path, grs1758_flux)

grs1758 = PointSource(
    "grs_1758_258",
    l=4.50780,
    b=-1.36106,
    spectral_shape=grs1758_spectrum,
)

In [9]:
# Cen A: retain the injected power-law parameters.
cena_spectrum = Powerlaw()
cena_spectrum.K.value = 2.227338148135328e-6
cena_spectrum.piv.value = 300.0
cena_spectrum.index.value = -1.732

cena_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
cena_spectrum.piv.unit = u.keV
cena_spectrum.piv.fix = True
cena_spectrum.index.fix = True
cena = PointSource(
    "cena",
    l=309.516,
    b=19.417,
    spectral_shape=cena_spectrum,
)

In [10]:
# 4C 71.07: retain the injected spectral table.
four_c_71p07_spectrum = table_spectrum(four_c_71p07_path, 0.00119)
four_c_71p07 = PointSource(
    "four_c_71p07",
    l=143.540759,
    b=34.425671,
    spectral_shape=four_c_71p07_spectrum,
)

In [11]:
# 4C 21.35: keep non-flare and flare as separate catalog entries.
four_c_21p35_noflare_spectrum = table_spectrum(
    four_c_21p35_noflare_path,
    0.0005535,
)
four_c_21p35_noflare = PointSource(
    "four_c_21p35_noflare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_noflare_spectrum,
)

four_c_21p35_flare_spectrum = table_spectrum(
    four_c_21p35_flare_path,
    0.017293201590129727,
)
four_c_21p35_flare = PointSource(
    "four_c_21p35_flare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_flare_spectrum,
)

In [12]:
# MAXI J1820: combine the two injected energy-range tables.
maxi_j1820_low = table_spectrum(
    maxi_j1820_low_path,
    0.13820979415525897,
)
maxi_j1820_high = table_spectrum(
    maxi_j1820_high_path,
    0.005963517352694539,
)
maxi_j1820 = PointSource(
    "maxi_j1820",
    l=35.8535,
    b=10.15915,
    spectral_shape=maxi_j1820_low + maxi_j1820_high,
)

In [13]:
# MAXI J1348−630: combine the two injected energy-range tables.
maxi_j1348_low = table_spectrum(
    maxi_j1348_low_path,
    0.08633295828868433,
)
maxi_j1348_high = table_spectrum(
    maxi_j1348_high_path,
    0.0023113717669982913,
)
maxi_j1348 = PointSource(
    "maxi_j1348",
    l=309.26389732,
    b=-1.10328,
    spectral_shape=maxi_j1348_low + maxi_j1348_high,
)

In [14]:
# 3C 454.3: keep low and high states as separate entries.
three_c_454p3_low = PointSource(
    "three_c_454p3_low",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_low_path, 0.00029),
)
three_c_454p3_high = PointSource(
    "three_c_454p3_high",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_high_path, 0.00624),
)

In [15]:
# NGC 1068: read the injected table and position.
definition = find_source_file("DC4", "NGC_1068.source")
properties = {}
for raw_line in definition.read_text().splitlines():
    fields = raw_line.split("#", 1)[0].split()
    if fields and "." in fields[0]:
        properties[fields[0].split(".", 1)[1]] = fields[1:]
assert properties["Spectrum"][0].lower() == "file"
spectrum = table_spectrum(
    definition.parent / properties["Spectrum"][1], float(properties["Flux"][0]),
)
latitude, longitude = map(float, properties["Orientation"][-2:])
ngc1068 = PointSource("ngc1068", l=longitude % 360.0, b=latitude, spectral_shape=spectrum)

In [16]:
# NGC 4151: use the updated CPL+PL injection.
ngc4151_thermal = Cutoff_powerlaw()
ngc4151_thermal.K.value = 0.15 * 200.0**-1.75
ngc4151_thermal.piv.value = 200.0
ngc4151_thermal.index.value = -1.75
ngc4151_thermal.xc.value = 200.0
ngc4151_tail = Powerlaw()
ngc4151_tail_fraction_at_200kev = 0.30
ngc4151_tail.K.value = (
    ngc4151_tail_fraction_at_200kev
    * ngc4151_thermal.evaluate_at(200.0)
)
ngc4151_tail.piv.value = 200.0
ngc4151_tail.index.value = -3.8

for component in (ngc4151_thermal, ngc4151_tail):
    component.K.unit = DIFFERENTIAL_FLUX_UNIT
    component.piv.unit = u.keV
    component.piv.fix = True
    component.index.fix = True

ngc4151_thermal.xc.unit = u.keV
ngc4151_thermal.xc.fix = True
assert np.isclose(
    ngc4151_tail.evaluate_at(200.0)
    / ngc4151_thermal.evaluate_at(200.0),
    ngc4151_tail_fraction_at_200kev,
)

ngc4151_energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
ngc4151_injected_flux = np.trapezoid(
    (ngc4151_thermal + ngc4151_tail)(ngc4151_energy),
    ngc4151_energy,
)
assert np.isclose(ngc4151_injected_flux, 0.0025171894, rtol=1e-5)

ngc4151 = PointSource(
    "ngc4151",
    l=155.077404,
    b=75.063170,
    spectral_shape=ngc4151_thermal + ngc4151_tail,
)

In [17]:
# Cyg X-3: preserve the transition-state table.
cyg_x3 = PointSource(
    "cyg_x3",
    l=79.84549,
    b=0.70006,
    spectral_shape=table_spectrum(cyg_x3_path, 0.001331405),
)

In [18]:
# PSR B1259−63: preserve the injected table.
psr_b1259 = PointSource(
    "psr_b1259",
    l=304.18358257,
    b=-0.99158457,
    spectral_shape=table_spectrum(psr_b1259_path, 0.00061253),
)

In [19]:
# 1RXS J170849.0−400901: read the injected table and position.
definition = find_source_file("DC4", "1RXSJ170849.source")
properties = {}
for raw_line in definition.read_text().splitlines():
    fields = raw_line.split("#", 1)[0].split()
    if fields and "." in fields[0]:
        properties[fields[0].split(".", 1)[1]] = fields[1:]
assert properties["Spectrum"][0].lower() == "file"
spectrum = table_spectrum(
    definition.parent / properties["Spectrum"][1], float(properties["Flux"][0]),
)
latitude, longitude = map(float, properties["Orientation"][-2:])
one_rxs_j170849 = PointSource("one_rxs_j170849", l=longitude % 360.0, b=latitude, spectral_shape=spectrum)

In [20]:
# Generic magnetar: read the injected table and position.
definition = find_source_file("DC4", "magnetar2.source")
properties = {}
for raw_line in definition.read_text().splitlines():
    fields = raw_line.split("#", 1)[0].split()
    if fields and "." in fields[0]:
        properties[fields[0].split(".", 1)[1]] = fields[1:]
assert properties["Spectrum"][0].lower() == "file"
spectrum = table_spectrum(
    definition.parent / properties["Spectrum"][1], float(properties["Flux"][0]),
)
latitude, longitude = map(float, properties["Orientation"][-2:])
generic_magnetar = PointSource("generic_magnetar", l=longitude % 360.0, b=latitude, spectral_shape=spectrum)

In [21]:
# LS 5039: preserve the injected table.
ls_5039 = PointSource(
    "ls_5039",
    l=16.88158741,
    b=-1.28921165,
    spectral_shape=table_spectrum(ls_5039_path, 0.0003006),
)

In [22]:
# PSR J1846−0258: preserve the injected table.
psr_j1846 = PointSource(
    "psr_j1846",
    l=29.71195,
    b=-0.24012,
    spectral_shape=table_spectrum(psr_j1846_path, 0.000152392),
)

In [23]:
# CO nova continuum: preserve the injected continuum table.
co_nova_continuum = PointSource(
    "co_nova_continuum",
    l=310.9847,
    b=2.7256,
    spectral_shape=table_spectrum(co_nova_continuum_path, 0.122071),
)

## 4. Parse the remaining MEGAlib source definitions

The parser supports the beam and spectrum types present in the 71-file manifest. For coupled spatial and energy maps, it points the YAML at an **existing** GALPROP-compatible template. A missing template raises an error; no new FITS file is written.

In [24]:
def safe_name(value):
    name = re.sub(r"[^A-Za-z0-9_]", "_", str(value)).strip("_").lower()
    if not name or name[0].isdigit():
        name = f"src_{name}"
    return name


def find_source_definition(challenge, source_stem):
    matches = sorted(
        (SOURCE_DATA_ROOT / challenge / "sources").rglob(
            f"{source_stem}.source"
        )
    )
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {source_stem}.source below {challenge}; "
            f"found {matches}"
        )
    return matches[0].resolve()


def resolve_source_asset(source_definition, filename):
    path = source_definition.parent / filename
    candidates = [path, Path(f"{path}.gz")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not resolve {filename!r} next to {source_definition}"
    )


def parse_megalib_blocks(source_definition):
    # Each DataChallenge.Source block becomes one component description.
    blocks = []
    by_megalib_name = {}
    for raw_line in source_definition.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        tokens = line.split()
        if tokens[:1] == ["DataChallenge.Source"]:
            megalib_name = tokens[1]
            block = {"_name": megalib_name}
            blocks.append(block)
            by_megalib_name[megalib_name] = block
            continue
        if "." not in tokens[0]:
            continue
        megalib_name, prop = tokens[0].split(".", 1)
        if megalib_name in by_megalib_name:
            by_megalib_name[megalib_name][prop] = tokens[1:]
    if not blocks:
        raise ValueError(f"No DataChallenge.Source blocks in {source_definition}")
    return blocks


def normalization_parameter(shape):
    candidates = [
        parameter
        for parameter in shape.parameters.values()
        if parameter.is_normalization
    ]
    if not candidates:
        # astromodels marks the Gaussian-line amplitude F as a prior
        # parameter rather than a source normalization.  It is still the
        # multiplicative integrated line flux for this use.
        candidates = [
            shape.parameters[name]
            for name in ("F", "K", "k")
            if name in shape.parameters
        ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one normalization in {shape.name}; found {candidates}"
        )
    return candidates[0]


def normalize_shape_in_band(shape, photon_flux):
    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    integral = float(np.trapezoid(shape(energy), energy))
    if not np.isfinite(integral) or integral <= 0:
        raise ValueError(f"Invalid integral {integral} for {shape}")
    normalization_parameter(shape).value *= float(photon_flux) / integral
    return shape


def make_parsed_spectrum(block, source_definition):
    tokens = block["Spectrum"]
    kind = tokens[0].lower()
    photon_flux = float(block["Flux"][0])

    if kind == "file":
        return table_spectrum(
            resolve_source_asset(source_definition, tokens[1]),
            photon_flux,
        )

    if kind == "gaussian":
        shape = Gaussian()
        shape.F.value = 1.0
        shape.mu.value = float(tokens[1])
        shape.sigma.value = float(tokens[2])
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "band":
        shape = Band()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.alpha.value = float(tokens[3])
        shape.beta.value = float(tokens[4])
        shape.xp.value = float(tokens[5])
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "comptonized":
        alpha = float(tokens[3])
        epeak = float(tokens[4])
        shape = Cutoff_powerlaw()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.index.value = alpha
        shape.xc.value = epeak / (2.0 + alpha)
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "powerlaw":
        shape = Powerlaw()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.index.value = -float(tokens[3])
        return normalize_shape_in_band(shape, photon_flux)

    raise NotImplementedError(
        f"Unsupported spectrum {block['Spectrum']} in {source_definition}"
    )


def make_polarization(block):
    tokens = block.get("Polarization")
    if tokens is None:
        return None
    # MEGAlib stores degree as a fraction; astromodels uses percent.
    polarization = LinearPolarization(
        100.0 * float(tokens[-2]),
        float(tokens[-1]),
    )
    polarization.degree.fix = True
    polarization.angle.fix = True
    return polarization

In [25]:
def make_map_source(unit_key, block, source_definition):
    """Attach an existing spatial/energy template to an extended source."""
    template_path = SPATIAL_TEMPLATE_DIRECTORY / (
        f"{unit_key}_nside{SPATIAL_TEMPLATE_NSIDE}_{SPATIAL_TEMPLATE_FORMAT_TAG}.fits"
    )
    if not template_path.is_file():
        raise FileNotFoundError(
            f"Missing {template_path}. Generate the spatial template with the full "
            "catalog notebook before running this YAML-only notebook."
        )
    # Verify that the MEGAlib map named in the source definition is present.
    resolve_source_asset(source_definition, block["Beam"][1])

    spatial = GalpropHealpixModel()
    spatial._fitsfile = str(template_path.resolve())
    spatial._file_loaded = False  # Defer loading the large map until fitting.
    spatial.K.value = 1.0
    source = ExtendedSource(unit_key, spatial_shape=spatial)
    # Full response folding sets this implicit Constant unit. Set it here because
    # this notebook deliberately skips response folding.
    source.components["main"].shape.k.unit = DIFFERENTIAL_FLUX_UNIT
    return source

In [26]:
def parse_megalib_source(unit_key, source_definition):
    # Build one astromodels source from one manifest simulation file.
    blocks = parse_megalib_blocks(source_definition)
    beam_kinds = {block["Beam"][0] for block in blocks}
    if beam_kinds == {"FarFieldNormalizedEnergyBeamFluxFunction"}:
        if len(blocks) != 1:
            raise ValueError(
                f"Expected one coupled map in {source_definition}; got {len(blocks)}"
            )
        return make_map_source(unit_key, blocks[0], source_definition)

    group_keys = []
    groups = {}
    for block in blocks:
        orientation = tuple(block["Orientation"][-2:])
        beam = tuple(block["Beam"])
        group_key = (beam, orientation)
        if group_key not in groups:
            groups[group_key] = []
            group_keys.append(group_key)
        groups[group_key].append(block)
    if len(groups) != 1:
        raise ValueError(
            f"{source_definition} contains multiple positions/morphologies. "
            "Add an explicit grouping rule before catalog generation."
        )

    (beam, orientation), grouped_blocks = next(iter(groups.items()))
    latitude, longitude = map(float, orientation)
    longitude %= 360.0
    component_names = []
    used_component_names = set()
    spectral_shapes = []
    polarizations = []
    for index, block in enumerate(grouped_blocks, start=1):
        component_name = safe_name(block["_name"])
        if component_name in used_component_names:
            component_name = f"{component_name}_{index}"
        used_component_names.add(component_name)
        component_names.append(component_name)
        spectral_shapes.append(make_parsed_spectrum(block, source_definition))
        polarizations.append(make_polarization(block))

    if beam[0] == "FarFieldPointSource":
        components = [
            SpectralComponent(name, shape, polarization)
            for name, shape, polarization in zip(
                component_names, spectral_shapes, polarizations
            )
        ]
        return PointSource(
            unit_key,
            l=longitude,
            b=latitude,
            components=components,
        )

    if beam[0] == "FarFieldGaussian":
        spatial = Gaussian_on_sphere(
            lon0=longitude,
            lat0=latitude,
            sigma=float(beam[-1]),
        )
    elif beam[0] == "FarFieldAssymetricGaussian":
        axis_a = float(beam[3])
        axis_b = float(beam[4])
        major = max(axis_a, axis_b)
        minor = min(axis_a, axis_b)
        eccentricity = np.sqrt(max(0.0, 1.0 - (minor / major) ** 2))
        spatial = Asymm_Gaussian_on_sphere(
            lon0=longitude,
            lat0=latitude,
            a=major,
            e=eccentricity,
            theta=float(beam[5]),
        )
    else:
        raise NotImplementedError(
            f"Unsupported beam {beam} in {source_definition}"
        )

    # BinnedThreeMLExtendedSourceResponse folds spectrum.main. Collapse
    # co-spatial line/continuum components into one composite spectral shape
    # so every analytic extended source has that standard component path.
    polarization_signatures = [
        None if polarization is None else polarization.to_dict()
        for polarization in polarizations
    ]
    if any(
        signature != polarization_signatures[0]
        for signature in polarization_signatures[1:]
    ):
        raise ValueError(
            f"{source_definition} has co-spatial components with different "
            "polarizations, which cannot be collapsed into one extended source"
        )
    spectral_shape = spectral_shapes[0]
    for component_shape in spectral_shapes[1:]:
        spectral_shape = spectral_shape + component_shape
    return ExtendedSource(
        unit_key,
        spatial_shape=spatial,
        spectral_shape=spectral_shape,
        polarization=polarizations[0],
    )

## 5. Build the 71-source model

This manifest keeps only the source keys and `.source` identifiers needed for YAML generation. Event FITS filenames are omitted because the catalog values do not use event counts.

In [27]:
BASE_SIMULATION_UNITS = [
    ('cyg_x1_hard', cygx1),
    ('crab', crab),
    ('one_e_1740_compow', one_e_1740),
    ('grs_1758_258', grs1758),
    ('cena', cena),
    ('four_c_71p07', four_c_71p07),
    ('four_c_21p35_noflare', four_c_21p35_noflare),
    ('four_c_21p35_flare', four_c_21p35_flare),
    ('maxi_j1820', maxi_j1820),
    ('maxi_j1348', maxi_j1348),
    ('three_c_454p3_low', three_c_454p3_low),
    ('three_c_454p3_high', three_c_454p3_high),
    ('ngc1068', ngc1068),
    ('ngc4151', ngc4151),
    ('cyg_x3', cyg_x3),
    ('psr_b1259', psr_b1259),
    ('one_rxs_j170849', one_rxs_j170849),
    ('generic_magnetar', generic_magnetar),
    ('ls_5039', ls_5039),
    ('psr_j1846', psr_j1846),
    ('co_nova_continuum', co_nova_continuum),
]

PARSED_SIMULATION_UNITS = [
    ('tuc_47', 'DC3', 'Globular_Cluster_Tuc_47'),
    ('omega_cen', 'DC3', 'Globular_Cluster_Omega_Cen'),
    ('ngc_6397', 'DC3', 'Globular_Cluster_NGC_6397'),
    ('ngc_6121', 'DC3', 'Globular_Cluster_NGC_6121'),
    ('positrons_26al_line', 'DC3', 'Positrons_from_26Al_line'),
    ('positrons_26al_cont', 'DC3', 'Positrons_from_26Al_cont'),
    ('positrons_44ti_line', 'DC3', 'Positrons_from_44Ti_line'),
    ('positrons_44ti_cont', 'DC3', 'Positrons_from_44Ti_cont'),
    ('narrow_bulge_511', 'DC3', 'NB_511'),
    ('broad_bulge_511', 'DC3', 'BB_511'),
    ('vela_snr_511', 'DC3', 'Vela_SNR_511'),
    ('co_nova_511kev', 'DC3', 'nova_co_511keV'),
    ('co_nova_478kev', 'DC3', 'nova_co_478keV'),
    ('al26_cyg_region', 'DC3', '26Al_Cyg_Region'),
    ('al26_ne2001', 'DC3', '26Al_NE2001'),
    ('fe60_cyg_region', 'DC3', '60Fe_Cyg_Region'),
    ('fe60_ne2001', 'DC3', '60Fe_NE2001'),
    ('grb_bn081207680', 'DC3', 'bn081207680'),
    ('grb_bn090424592', 'DC3', 'bn090424592'),
    ('grb_bn100612726', 'DC3', 'bn100612726'),
    ('grb_bn110605183', 'DC3', 'bn110605183'),
    ('grb_bn131122490', 'DC3', 'bn131122490'),
    ('grb_bn140329295', 'DC3', 'bn140329295'),
    ('grb_bn161004964', 'DC3', 'bn161004964'),
    ('grb_bn170405777', 'DC3', 'bn170405777'),
    ('grb_bn180504136', 'DC3', 'bn180504136'),
    ('grb_bn180703876', 'DC3', 'bn180703876'),
    ('grb_bn080802386_flux150', 'DC3', 'bn080802386'),
    ('mgf_051103', 'DC3', 'MGF051103'),
    ('mgf_070201', 'DC3', 'MGF070201'),
    ('mgf_070222', 'DC3', 'MGF070222'),
    ('mgf_180128a', 'DC3', 'MGF180128A'),
    ('mgf_200415a', 'DC3', 'MGF200415A'),
    ('mgf_231115a', 'DC3', 'MGF231115A'),
    ('casa_g16_distribution', 'DC4', 'CasAG16distribution'),
    ('g1903', 'DC4', 'G1903'),
    ('kepler', 'DC4', 'Kepler'),
    ('mgt_burst_bright_complex', 'DC3', 'MgtBurst_bright_complex'),
    ('orion_eridanus', 'DC4', 'OrEr'),
    ('positron_annihilation_in_flight', 'DC4', 'Positrons_In_Flight_Annihilation'),
    ('positrons_central_source', 'DC4', 'Positrons_Central_Source'),
    ('positrons_thin_disk_cont', 'DC4', 'Positrons_Thin_Disk_cont'),
    ('positrons_thin_disk_line', 'DC4', 'Positrons_Thin_Disk_line'),
    ('sn1987a', 'DC4', 'SN1987A'),
    ('tycho', 'DC4', 'Tycho'),
    ('upper_scorpius', 'DC4', 'USco'),
    ('vela_jr_44ti', 'DC4', 'VelaJr_44Ti'),
    ('lmc_511', 'DC3', 'LMC_Gaussian_511'),
    ('m31_511', 'DC3', 'M31_Gaussian_511'),
    ('virgo_511', 'DC3', 'Virgo_Gaussian_511'),
]

assert len(BASE_SIMULATION_UNITS) == 21
assert len(PARSED_SIMULATION_UNITS) == 50

In [28]:
simulation_units = {}
for key, source in BASE_SIMULATION_UNITS:
    simulation_units[key] = source
for key, challenge, stem in PARSED_SIMULATION_UNITS:
    source_definition = find_source_definition(challenge, stem)
    simulation_units[key] = parse_megalib_source(key, source_definition)
if len(simulation_units) != 71:
    raise RuntimeError(f"Expected 71 simulation entries, found {len(simulation_units)}")

catalog = Model(*simulation_units.values())
if len(catalog.sources) != 71:
    raise RuntimeError("The catalog must contain 71 distinct sources")

In [29]:
BASE_SOURCE_DEFINITIONS = {
    "cyg_x1_hard": ("DC3", "cygX1_hard"),
    "crab": ("DC4", "Crab"),
    "one_e_1740_compow": ("DC3", "1E1740_compow"),
    "grs_1758_258": ("DC3", "GRS1758"),
    "cena": ("DC4", "CenA"),
    "four_c_71p07": ("DC4", "4C71p07"),
    "four_c_21p35_noflare": ("DC3", "4C21p35_noflare"),
    "four_c_21p35_flare": ("DC3", "4C21p35_flare"),
    "maxi_j1820": ("DC3", "MAXIJ1820"),
    "maxi_j1348": ("DC3", "MAXIJ1348"),
    "three_c_454p3_low": ("DC4", "3C454p3_low"),
    "three_c_454p3_high": ("DC4", "3C454p3_high"),
    "ngc1068": ("DC4", "NGC_1068"),
    "ngc4151": ("DC4", "NGC_4151"),
    "cyg_x3": ("DC4", "cygX3"),
    "psr_b1259": ("DC3", "PSRB1259"),
    "one_rxs_j170849": ("DC4", "1RXSJ170849"),
    "generic_magnetar": ("DC4", "magnetar2"),
    "ls_5039": ("DC3", "LS5039"),
    "psr_j1846": ("DC3", "PSRJ1846"),
    "co_nova_continuum": ("DC3", "nova_co_continuum"),
}
source_definitions = {
    key: find_source_definition(*definition)
    for key, definition in BASE_SOURCE_DEFINITIONS.items()
}
source_definitions.update({
    key: find_source_definition(challenge, stem)
    for key, challenge, stem in PARSED_SIMULATION_UNITS
})
if set(source_definitions) != set(simulation_units):
    raise RuntimeError("Source definitions do not match the manifest")
injection_blocks = {key: parse_megalib_blocks(path) for key, path in source_definitions.items()}

## 6. Apply both full-observation lightcurve means

For each component, integrate its piecewise-linear lightcurve over the full observation and divide by the elapsed duration:

`catalog amplitude = injected amplitude × ∫ L(t) dt / (t_stop − t_start)`

The **parallel-chunks** catalog uses 1000 simulation bins for mission-grid, non-repeating profiles, except the CO nova continuum's inferred 500-bin setting. Other lightcurves use global normalization. The **no-chunks** catalog uses one global normalization interval for every variable lightcurve. Steady sources keep their injected amplitudes in both catalogs.

The nova bin count is an inference from earlier event-time comparisons, so verify it against archived simulation settings if those become available. No FITS counts enter this calculation.


In [30]:
# The parallel convention depends on the lightcurve sampling grid.
PARALLEL_SIMULATION_BINS = 1000
PARALLEL_SIMULATION_BINS_BY_SOURCE = {"co_nova_continuum": 500}
LIGHTCURVE_MODE_BY_SOURCE = {}  # Optional explicit per-source overrides.

def read_lightcurve(path):
    """Read plain or gzipped MEGAlib `DP time rate` rows."""
    opener = gzip.open if Path(path).suffix == ".gz" else open
    times, rates = [], []
    with opener(path, "rt") as stream:
        for line in stream:
            fields = line.split("#", 1)[0].split()
            if not fields:
                continue
            if fields[0] == "IP" and (len(fields) != 2 or fields[1].upper() != "LINLIN"):
                raise ValueError(f"Unsupported lightcurve interpolation in {path}")
            if fields[0] == "DP":
                if len(fields) < 3:
                    raise ValueError(f"Malformed lightcurve row in {path}: {line.strip()}")
                times.append(float(fields[1]))
                rates.append(float(fields[2]))
    return np.asarray(times), np.asarray(rates)

def lightcurve_recipe(key, times, repeating, history, method):
    """Select the requested MEGAlib normalization convention."""
    if method == "no_chunks":
        return "global_megalib", 1
    if method != "parallel_chunks":
        raise ValueError(f"Unknown normalization method: {method}")
    mission_grid = (
        len(times) == len(history.obstime)
        and np.allclose(times, history.obstime.unix, rtol=0.0, atol=1e-5)
    )
    default_mode = "parallel_megalib" if mission_grid and not repeating else "global_megalib"
    mode = LIGHTCURVE_MODE_BY_SOURCE.get(key, default_mode)
    bins = (PARALLEL_SIMULATION_BINS_BY_SOURCE.get(key, PARALLEL_SIMULATION_BINS)
            if mode == "parallel_megalib" else 1)
    return mode, bins

In [31]:
def source_amplitudes(source):
    """Find the multiplicative amplitudes to scale for one source."""
    spatial_shape = getattr(source, "spatial_shape", None)
    if isinstance(spatial_shape, GalpropHealpixModel):
        return [spatial_shape.K]
    amplitudes = []
    for component in source.components.values():
        shape_amplitudes = [p for p in component.shape.parameters.values() if p.is_normalization]
        if not shape_amplitudes:
            # Gaussian line F is not flagged as a normalization by astromodels.
            shape_amplitudes = [p for p in component.shape.parameters.values()
                                if p.name.split("_", 1)[0] in {"F", "K", "k"}]
        amplitudes.extend(shape_amplitudes)
    if not amplitudes:
        raise RuntimeError(f"No source amplitude found for {source.name}")
    return amplitudes

def temporal_groups(key, source, blocks):
    """Map source-file lightcurves to astromodels component amplitudes."""
    if len({tuple(block.get("Lightcurve", [])) for block in blocks}) == 1:
        return [("all", blocks[0], source_amplitudes(source))]
    groups = []
    for block in blocks:
        matches = []
        for name, component in source.components.items():
            if name == safe_name(block["_name"]):
                matches.append(component)
            elif block.get("Spectrum", [""])[0].lower() == "file" and isinstance(component.shape, SpecFromDat):
                if Path(component.shape.dat.value).name == block["Spectrum"][1]:
                    matches.append(component)
        if len(matches) != 1:
            raise ValueError(f"Cannot uniquely map lightcurve for {key}/{block['_name']}")
        shape = matches[0].shape
        amplitudes = [p for p in shape.parameters.values() if p.is_normalization]
        if not amplitudes:
            amplitudes = [normalization_parameter(shape)]
        groups.append((block["_name"], block, amplitudes))
    return groups

In [32]:
# Use elapsed observation time, matching the full catalog and response recipe.
full_history = SpacecraftHistory.open(ORIENTATION_PATH)
start = float(full_history.obstime[0].unix)
stop = float(full_history.obstime[-1].unix)
duration = stop - start
if duration <= 0:
    raise ValueError("Orientation has no positive elapsed duration")

def build_time_averaged_catalog(seed_catalog, method):
    """Scale a fresh model copy so the two methods cannot affect each other."""
    result = deepcopy(seed_catalog)
    for key, blocks in injection_blocks.items():
        source = result.sources[key]
        for _, block, amplitudes in temporal_groups(key, source, blocks):
            lc = block.get("Lightcurve")
            factor = 1.0  # A steady source keeps its injected amplitude.
            if lc is not None:
                if len(lc) != 3 or lc[0].lower() != "file" or lc[1].lower() not in {"true", "false"}:
                    raise ValueError(f"Unsupported lightcurve for {key}: {lc}")
                path = resolve_source_asset(source_definitions[key], lc[2])
                times, rates = read_lightcurve(path)
                repeating = lc[1].lower() == "true"
                mode, bins = lightcurve_recipe(key, times, repeating, full_history, method)
                profile = LightcurveProfile(times, rates, repeating, mode, bins)
                factor = float(profile.integral([start], [stop])[0] / duration)
                if not np.isfinite(factor) or factor < 0:
                    raise ValueError(f"Invalid time factor for {key}: {factor}")
                # Preserve a tiny positive amplitude for sources outside the window.
                factor = max(factor, 1e-30)
            for amplitude in amplitudes:
                value = float(amplitude.value) * factor
                if not np.isfinite(value) or value < 0:
                    raise ValueError(f"Invalid amplitude for {key}: {value}")
                if amplitude.min_value is not None and value < amplitude.min_value:
                    amplitude.min_value = value / 10.0
                if amplitude.max_value is not None and value > amplitude.max_value:
                    amplitude.max_value = 10.0 ** np.ceil(np.log10(value))
                amplitude.value = value
                set_normalization_step(amplitude)
    return result

# `catalog` is the unscaled seed model built above. Keep it unchanged.
parallel_catalog = build_time_averaged_catalog(catalog, "parallel_chunks")
no_chunks_catalog = build_time_averaged_catalog(catalog, "no_chunks")


## 7. Freeze and save both catalogs

Each YAML contains 71 sources and no fitted/free parameters. The output filenames identify the normalization method. Use each YAML with a response recipe calculated using that same method; this notebook does not write response recipes.


In [33]:
catalogs_to_save = {
    "parallel_chunks": parallel_catalog,
    "no_chunks": no_chunks_catalog,
}

# Validate both models and paths before writing either file.
for method, output_catalog in catalogs_to_save.items():
    for parameter in output_catalog.parameters.values():
        if not parameter.has_auxiliary_variable:
            parameter.fix = True
    if output_catalog.free_parameters:
        raise RuntimeError(f"The {method} catalog must have no free parameters")
    if len(output_catalog.sources) != 71:
        raise RuntimeError(f"The {method} catalog must contain 71 sources")
for output_path in OUTPUT_CATALOGS.values():
    if output_path.exists() and not OVERWRITE_CATALOGS:
        raise FileExistsError(f"Output exists: {output_path}")

for method, output_catalog in catalogs_to_save.items():
    output_path = OUTPUT_CATALOGS[method]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_catalog.save(output_path, overwrite=OVERWRITE_CATALOGS)
    print(f"Saved {len(output_catalog.sources)} {method} sources to {output_path}")


Saved 71 parallel_chunks sources to /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/source_catalog_DC4_all_71files_full_elapsed_lightcurve_parallel_chunks_generated.yaml
Saved 71 no_chunks sources to /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/source_catalog_DC4_all_71files_full_elapsed_lightcurve_no_chunks_generated.yaml
